In [1]:
import torch
print("PyTorch:", torch.__version__, "GPU available:", torch.cuda.is_available())
import torchvision
print("Torchvision:", torchvision.__version__)

PyTorch: 2.11.0+cu130 GPU available: True
Torchvision: 0.26.0+cu130


In [2]:
from __future__ import annotations

import argparse
import copy
import csv
import heapq
import json
import math
import os
import platform
import random
import sys
import time
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset, TensorDataset

# Only import torchvision when loading a public image dataset, so the offline
# synthetic smoke test does not require torchvision or internet access.

METHODS = ("dwafl", "fedavg", "fedasync", "lag", "dgc", "paflm")
ABLATIONS = ("no_dual", "no_compression", "no_resource", "no_async", "no_error_feedback", "no_noise")


class CNNGRU(nn.Module):
    """CNN+GRU model, with parameter groups for resource-aware partial training."""

    def __init__(self, channels=1, width=16):
        super().__init__()
        self.conv1 = nn.Sequential(nn.Conv2d(channels, width, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.conv2 = nn.Sequential(nn.Conv2d(width, width, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d((4, 4)))
        self.gru = nn.GRU(width * 4, 48, batch_first=True)
        self.head = nn.Linear(48, 10)

    def forward(self, x):
        x = self.conv2(self.conv1(x))  # N,C,4,4
        x = x.permute(0, 2, 1, 3).contiguous().reshape(x.size(0), 4, -1)
        return self.head(self.gru(x)[1][-1])


def seed_everything(seed, deterministic=False):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        try:
            torch.use_deterministic_algorithms(True, warn_only=True)
        except TypeError:
            torch.use_deterministic_algorithms(True)


def load_data(name, root, n_train, n_test, seed):
    if name == "synthetic":
        # Synthetic data are for an OFFLINE pipeline test, never journal results.
        gen = torch.Generator().manual_seed(seed + 101)

        def make(n):
            y = torch.arange(n, dtype=torch.long) % 10
            x = .12 * torch.randn(n, 1, 28, 28, generator=gen)
            for j in range(n):
                label = int(y[j])
                x[j, 0, 2 + 2 * label:4 + 2 * label, 4:24] += 1.8
            ds = TensorDataset(x, y)
            ds.targets = y.tolist()
            return ds
        return make(n_train or 200), make(n_test or 100), 1

    from torchvision import datasets, transforms
    if name == "mnist":
        transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((.1307,), (.3081,))])
        dataset_class, channels = datasets.MNIST, 1
    else:
        transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((.4914, .4822, .4465), (.247, .243, .261))])
        dataset_class, channels = datasets.CIFAR10, 3
    train = dataset_class(root, train=True, transform=transform, download=True)
    test = dataset_class(root, train=False, transform=transform, download=True)
    rng = np.random.default_rng(seed)
    if n_train and n_train < len(train):
        train = Subset(train, sorted(rng.choice(len(train), n_train, replace=False).tolist()))
    if n_test and n_test < len(test):
        test = Subset(test, sorted(rng.choice(len(test), n_test, replace=False).tolist()))
    return train, test, channels


def dataset_labels(ds):
    if isinstance(ds, Subset):
        return np.asarray(ds.dataset.targets, dtype=np.int64)[ds.indices]
    return np.asarray(ds.targets, dtype=np.int64)


def partition_clients(ds, n_clients, seed, mode="dirichlet", alpha=.5, min_samples=2):
    """Reproducible label-skew partitioning, with minimum-sample repair.

    For severe skew, pure Dirichlet allocations often leave empty clients. We
    transfer surplus examples from larger clients to deficient clients (and
    disclose that repair); every training example appears exactly once.
    """
    y = dataset_labels(ds)
    if len(y) < n_clients * min_samples:
        raise ValueError(f"Dataset has {len(y)} samples, cannot give {min_samples} to {n_clients} clients")
    rng = np.random.default_rng(seed)
    if mode == "iid":
        return [part.tolist() for part in np.array_split(rng.permutation(len(y)), n_clients)]
    if alpha <= 0:
        raise ValueError("Dirichlet alpha must be > 0")
    parts = [[] for _ in range(n_clients)]
    for category in np.unique(y):
        ids = rng.permutation(np.where(y == category)[0])
        proportions = rng.dirichlet(np.full(n_clients, alpha))
        counts = rng.multinomial(len(ids), proportions)
        boundaries = np.cumsum(counts)[:-1]
        for cid, chunk in enumerate(np.split(ids, boundaries)):
            parts[cid].extend(int(i) for i in chunk)
    for small in sorted(range(n_clients), key=lambda k: len(parts[k])):
        while len(parts[small]) < min_samples:
            big = max(range(n_clients), key=lambda k: len(parts[k]))
            if len(parts[big]) <= min_samples:
                raise RuntimeError("Unable to rebalance non-IID data partition")
            loc = int(rng.integers(0, len(parts[big])))
            parts[small].append(parts[big].pop(loc))
    return [rng.permutation(part).tolist() for part in parts]


def flat_state(state):
    return torch.cat([v.detach().reshape(-1).to(device="cpu", dtype=torch.float32) for v in state.values()])


def unflat_state(vec, reference):
    ans = {}
    pos = 0
    for key, val in reference.items():
        amount = val.numel()
        ans[key] = vec[pos:pos+amount].reshape(val.shape).to(device=val.device, dtype=val.dtype).clone()
        pos += amount
    if pos != vec.numel():
        raise ValueError("Model vector and reference do not have equal length")
    return ans


def state_template(model):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def make_masks(reference):
    # Each tier trains/communicates the complete selected parameter tensors.
    groups = {"light": ("head.", "gru."), "medium": ("head.", "gru.", "conv2."), "full": ("head.", "gru.", "conv2.", "conv1.")}
    masks = {}
    for tier, prefixes in groups.items():
        vals = []
        for k, v in reference.items():
            vals.append(torch.full((v.numel(),), any(k.startswith(p) for p in prefixes), dtype=torch.bool))
        masks[tier] = torch.cat(vals)
    return masks


class ResourcePolicy(nn.Module):
    """GRU policy trained via REINFORCE on a simulated resource-feasibility reward.

    This is an explicit model-based simulation of resource decisions, NOT
    evidence of RL results from physical edge nodes.
    """
    def __init__(self):
        super().__init__()
        self.gru = nn.GRU(3, 16, batch_first=True)
        self.head = nn.Linear(16, 3)

    def forward(self, history):
        return self.head(self.gru(history)[1][-1])


def resource_reward(state, action):
    capacity = (.40 * state[:, 0] + .35 * state[:, 1] + .25 * state[:, 2]).clamp(0, 1)
    utilization = torch.tensor([.30, .60, 1.], device=state.device)[action]
    # Encourage larger workloads when safe, but penalize overload sharply.
    return utilization - 3.0 * torch.relu(utilization - capacity).square()


def fit_resource_policy(seed, steps=80):
    # Train once per run on synthetic histories, separate from private image data.
    gen = torch.Generator().manual_seed(seed + 701)
    model = ResourcePolicy()
    opt = torch.optim.Adam(model.parameters(), lr=.01)
    baseline = 0.
    for _ in range(steps):
        history = torch.rand((64, 5, 3), generator=gen)
        logits = model(history)
        probs = torch.softmax(logits, dim=1)
        actions = torch.multinomial(probs, num_samples=1, generator=gen).squeeze(1)
        rewards = resource_reward(history[:, -1], actions)
        baseline = .9 * baseline + .1 * float(rewards.mean().item())
        logprob = torch.log(probs.gather(1, actions[:, None]).squeeze(1).clamp_min(1e-12))
        entropy = -(probs * probs.clamp_min(1e-12).log()).sum(1)
        loss = -(logprob * (rewards - baseline).detach()).mean() - .01 * entropy.mean()
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
        opt.step()
    model.eval()
    return model


def choose_tier(policy, hist):
    with torch.no_grad():
        logits = policy(torch.as_tensor(hist, dtype=torch.float32).unsqueeze(0))
    return ("light", "medium", "full")[int(torch.argmax(logits, dim=1).item())]


def train_local(reference, factory, train_ds, ids, epochs, batch_size, lr, device, seed, tier="full"):
    net = factory().to(device)
    net.load_state_dict(reference)
    for k, p in net.named_parameters():
        if tier == "light" and k.startswith(("conv1.", "conv2.")):
            p.requires_grad_(False)
        elif tier == "medium" and k.startswith("conv1."):
            p.requires_grad_(False)
    params = [p for p in net.parameters() if p.requires_grad]
    optimizer = torch.optim.Adam(params, lr=lr)
    loader = DataLoader(Subset(train_ds, ids), batch_size=batch_size, shuffle=True,
                        num_workers=0, generator=torch.Generator().manual_seed(seed))
    net.train()
    correct, count, loss_total = 0, 0, 0.
    t0 = time.perf_counter()
    for _ in range(epochs):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = net(x)
            loss = nn.functional.cross_entropy(logits, y)
            loss.backward()
            optimizer.step()
            correct += int((logits.argmax(dim=1) == y).sum().item())
            count += int(y.numel())
            loss_total += float(loss.item() * y.numel())
    delta = flat_state(state_template(net)) - flat_state(reference)
    # Local minibatch training accuracy is only a quality heuristic.
    return delta, correct / max(1, count), time.perf_counter() - t0, loss_total / max(1, count)


@dataclass
class SparseMessage:
    # Sparse payload: one int32 index and one float32 value per retained entry.
    indices: torch.Tensor | None
    values: torch.Tensor
    length: int
    bytes_sent: int

    def decode(self):
        if self.indices is None:
            return self.values.clone()
        out = torch.zeros(self.length, dtype=torch.float32)
        if len(self.indices):
            out[self.indices.long()] = self.values
        return out


def exact_topk_message(vector, keep_ratio, allowed_mask=None):
    """Send exactly ceil(keep_ratio * allowed_count) COORDINATES.

    Avoid quantile thresholds because zero ties caused the old code to send
    every parameter at a nominal 50% keep rate.
    """
    if allowed_mask is None:
        allowed = torch.arange(vector.numel())
    else:
        allowed = torch.nonzero(allowed_mask, as_tuple=False).flatten()
    n = int(allowed.numel())
    if n == 0:
        return SparseMessage(torch.empty(0, dtype=torch.int32), torch.empty(0), vector.numel(), 12)
    k = max(1, min(n, math.ceil(keep_ratio * n)))
    if k == n and n == vector.numel():
        return SparseMessage(None, vector.detach().clone().float(), vector.numel(), 4 * vector.numel())
    indices = allowed[torch.topk(vector[allowed].abs(), k=k, largest=True, sorted=False).indices]
    indices = indices.sort().values.to(torch.int32)
    return SparseMessage(indices, vector[indices.long()].clone().to(torch.float32), vector.numel(), 12 + 8 * k)


def dense_message(vector):
    return SparseMessage(None, vector.clone().float(), vector.numel(), int(vector.numel() * 4))


def adaptive_ratio(base_ratio, ema, signal, decay, min_keep, max_keep):
    """When gradients change, retain more coordinates; when stable, retain fewer."""
    ema = signal if ema is None else decay * ema + (1 - decay) * signal
    relative = signal / max(ema, 1e-12)
    ratio = base_ratio * (relative ** .5)
    return float(np.clip(ratio, min_keep, max_keep)), float(ema)


def compress_update(delta, residual, method, keep_ratio, mask, use_error_feedback, noise_scale, clip, seed):
    """Clip before compression; compute error-feedback residual on the CLEAN payload.

    Noise is only an experimental perturbation and does NOT imply (epsilon,delta)-DP.
    """
    raw = delta.detach().clone().float()
    raw[~mask] = 0
    if clip > 0:
        norm = raw.norm().item()
        if norm > clip:
            raw *= clip / (norm + 1e-12)
    corrected = raw + (residual if use_error_feedback else 0.)
    corrected[~mask] = 0
    if method in ("dwafl", "dgc", "paflm"):
        msg = exact_topk_message(corrected, keep_ratio, mask)
    else:
        msg = dense_message(corrected)
    clean = msg.decode()
    new_residual = corrected - clean if use_error_feedback else torch.zeros_like(corrected)
    new_residual[~mask] = 0
    if noise_scale > 0:
        gen = torch.Generator().manual_seed(seed)
        # Perturb only transmitted values, keeping indices private only to the
        # extent of sparsity. No formal differential privacy protection.
        new_values = msg.values + torch.randn(msg.values.shape, generator=gen) * noise_scale
        msg = SparseMessage(msg.indices, new_values, msg.length, msg.bytes_sent)
    return msg, new_residual


@torch.no_grad()
def evaluate(model, dataset, device, batch=128):
    model.eval()
    cm = np.zeros((10, 10), dtype=np.int64)
    loss_sum, n = 0., 0
    for x, y in DataLoader(dataset, batch_size=batch):
        logits = model(x.to(device))
        y = y.to(device)
        loss_sum += float(nn.functional.cross_entropy(logits, y, reduction="sum").item())
        pred = logits.argmax(1).cpu().numpy()
        np.add.at(cm, (y.cpu().numpy(), pred), 1)
        n += int(y.numel())
    true_pos = cm.diagonal()
    precision = true_pos / np.maximum(1, cm.sum(0))
    recall = true_pos / np.maximum(1, cm.sum(1))
    f1 = 2 * precision * recall / np.maximum(1e-12, precision + recall)
    return {"accuracy": float(true_pos.sum() / max(n, 1)), "precision_macro": float(precision.mean()),
            "recall_macro": float(recall.mean()), "f1_macro": float(f1.mean()), "loss": float(loss_sum / max(n, 1))}


def write_csv(path, rows):
    if not rows:
        return
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def run_experiment(args, method, seed, ablation="full"):
    if method not in METHODS:
        raise ValueError(f"Unknown method: {method}")
    seed_everything(seed, args.deterministic)
    ds_train, ds_test, channels = load_data(args.dataset, args.data_dir, args.train_limit, args.test_limit, seed)
    parts = partition_clients(ds_train, args.clients, seed + 777, args.partition, args.alpha, args.min_samples)
    factory = lambda: CNNGRU(channels, args.width)
    model = factory().to(args.device)
    global_state = state_template(model)
    n_params = flat_state(global_state).numel()
    masks = make_masks(global_state)
    all_mask = masks["full"]
    out = Path(args.output) / args.dataset / method / f"seed_{seed}" / ablation
    out.mkdir(parents=True, exist_ok=True)
    (out / "partition.json").write_text(json.dumps({"indices": parts, "alpha": args.alpha,
                                                   "partition": args.partition, "seed": seed + 777}))
    (out / "configuration.json").write_text(json.dumps({**vars(args), "method": method, "seed": seed,
                                                        "ablation": ablation, "torch": torch.__version__,
                                                        "platform": platform.platform(), "python": sys.version}, indent=2))
    policy = fit_resource_policy(seed, args.policy_steps) if method == "dwafl" and ablation != "no_resource" else None
    resource_history = np.random.default_rng(seed + 11).uniform(.1, 1., (args.clients, 5, 3))
    ema = [None] * args.clients
    residuals = [torch.zeros(n_params) for _ in range(args.clients)]
    previous_sent = [None] * args.clients
    previous_round = [0] * args.clients
    cumulative_bytes = 0
    rows, pending = [], []
    packet_order = 0
    global_vec = flat_state(global_state)
    # Use independent RNGs per round; all methods get the SAME selected client IDs.
    # Delays only apply to async methods; number of training sessions stays equal.
    async_method = method in ("dwafl", "fedasync", "paflm") and ablation != "no_async"
    use_dual = method == "dwafl" and ablation != "no_dual"
    compression_method = method if ablation != "no_compression" else "fedavg"
    error_feedback = method in ("dwafl", "dgc") and ablation != "no_error_feedback"
    noise = args.noise if method == "dwafl" and ablation != "no_noise" else 0.

    def apply_arrivals(arrivals, current_round):
        nonlocal global_vec
        if not arrivals:
            return 0, 0., 0.
        ages = [max(0, current_round - packet["source_round"]) for packet in arrivals]
        if async_method:
            # Buffered asynchronous aggregation in a simulated event queue.
            # Updates are calculated from client-side historical snapshots, and
            # only arrivals due in this round contribute to the current model.
            # Missing parameter groups have per-coordinate normalization.
            # Crucially, we normalize by un-decayed sample weights, so quality
            # and staleness still attenuate even when only one update arrives.
            total = torch.zeros(n_params)
            denom = torch.zeros(n_params)
            for packet, age in zip(arrivals, ages):
                raw_weight = float(packet["samples"])
                quality = (.5 + .5 * packet["quality"]) if use_dual else 1.
                freshness = math.exp(-args.staleness_decay * age)
                weight = raw_weight * quality * freshness
                mask = packet["mask"]
                vec = packet["message"].decode()
                total[mask] += weight * vec[mask]
                denom[mask] += raw_weight
            have = denom > 0
            global_vec[have] += args.server_lr * total[have] / denom[have]
        else:
            # Synchronous weighted mean; normalize each parameter group only
            # over clients that actually trained/communicated that group.
            combined = torch.zeros(n_params)
            denom = torch.zeros(n_params)
            for packet in arrivals:
                weight = float(packet["samples"])
                if use_dual:
                    weight *= (.5 + .5 * packet["quality"])
                vec = packet["message"].decode()
                mask = packet["mask"]
                combined[mask] += vec[mask] * weight
                denom[mask] += weight
            have = denom > 0
            global_vec[have] += combined[have] / denom[have]
        return len(arrivals), float(np.mean(ages)), float(np.mean([p["network_ms"] for p in arrivals]))

    for rnd in range(1, args.rounds + 1):
        t_round = time.perf_counter()
        rng = np.random.default_rng(seed * 100000 + rnd)
        selected = rng.choice(args.clients, size=min(args.per_round, args.clients), replace=False).tolist()
        new_packets = []
        bytes_round = 0
        network_delays = []
        compute_times = []
        tiers = []
        # Decide all resource tiers before training. To prevent feature-extractor
        # starvation, ensure at least one selected client trains the full model.
        planned_tiers = {}
        resource_capacities = {}
        for cid in selected:
            resources = rng.uniform(.1, 1., size=3)
            resource_history[cid] = np.concatenate([resource_history[cid, 1:], resources[None]], axis=0)
            planned_tiers[cid] = choose_tier(policy, resource_history[cid]) if policy is not None else "full"
            resource_capacities[cid] = float(.40 * resources[0] + .35 * resources[1] + .25 * resources[2])
        if policy is not None and rnd <= args.warmup_rounds:
            # Train all CNN feature groups initially; partial subnetworks are
            # useful only after the global extractor is reasonably initialized.
            for cid in selected:
                planned_tiers[cid] = "full"
        if policy is not None and selected and all(planned_tiers[i] != "full" for i in selected):
            planned_tiers[max(selected, key=lambda i: resource_capacities[i])] = "full"
        for cid in selected:
            tier = planned_tiers[cid]
            tiers.append(tier)
            local_epochs = max(1, round(args.local_epochs * ({"light": .5, "medium": .75, "full": 1.}[tier])))
            # Identical local epochs for baseline algorithms and full-tier DW-AFL.
            delta, quality, cpu_s, train_loss = train_local(global_state, factory, ds_train, parts[cid], local_epochs,
                                                            args.batch, args.lr, args.device, seed + 30000 * rnd + cid, tier)
            mask = masks[tier]
            if method == "lag":
                if previous_sent[cid] is not None and (delta - previous_sent[cid]).norm().item() < args.lag_threshold:
                    continue
                previous_sent[cid] = delta.clone()
            # Adapt from CLIENT-specific signal, not a global/round constant.
            signal = float(delta[mask].abs().mean().item())
            ratio = args.keep_ratio
            if method == "dwafl" and ablation != "no_compression":
                ratio, ema[cid] = adaptive_ratio(args.keep_ratio, ema[cid], signal, args.ema_decay,
                                                  args.min_keep_ratio, args.max_keep_ratio)
            msg, residual = compress_update(delta, residuals[cid], compression_method, ratio, mask,
                                            error_feedback, noise, args.clip, seed + rnd * 1000 + cid)
            residuals[cid] = residual
            bytes_round += msg.bytes_sent
            bandwidth_bps = rng.uniform(args.bandwidth_min, args.bandwidth_max) * 1e6
            modeled_network_ms = args.propagation_ms + 1000. * 8. * msg.bytes_sent / bandwidth_bps
            network_delays.append(modeled_network_ms)
            compute_times.append(cpu_s)
            # Delays are actual queue ages in SIMULATED rounds, not wall-clock networking.
            delay_rounds = int(rng.integers(0, args.max_staleness + 1)) if async_method else 0
            packet = {"cid": cid, "source_round": rnd, "arrival_round": rnd + delay_rounds,
                      "message": msg, "mask": mask.clone(), "samples": len(parts[cid]),
                      "quality": quality, "network_ms": modeled_network_ms,
                      "local_loss": train_loss, "tier": tier}
            if async_method:
                packet_order += 1
                heapq.heappush(pending, (packet["arrival_round"], packet_order, packet))
            else:
                new_packets.append(packet)
        if async_method:
            while pending and pending[0][0] <= rnd:
                new_packets.append(heapq.heappop(pending)[2])
        applied, mean_stale, mean_net_arrivals = apply_arrivals(new_packets, rnd)
        global_state = unflat_state(global_vec, global_state)
        model.load_state_dict(global_state)
        metrics = evaluate(model, ds_test, args.device, args.eval_batch)
        cumulative_bytes += bytes_round
        row = {"round": rnd, "method": method, "seed": seed, "ablation": ablation,
               "train_clients": len(selected), "updates_sent": len(network_delays), "updates_applied": applied,
               "updates_queued": len(pending), "accuracy": metrics["accuracy"],
               "precision_macro": metrics["precision_macro"], "recall_macro": metrics["recall_macro"],
               "f1_macro": metrics["f1_macro"], "loss": metrics["loss"],
               "uplink_bytes": bytes_round, "uplink_bytes_cumulative": cumulative_bytes,
               "dense_uplink_bytes_selected": len(network_delays) * n_params * 4,
               "uplink_reduction_pct_vs_dense": 100. * (1. - bytes_round / max(1, len(network_delays)*n_params*4)),
               "simulated_network_ms_mean": float(np.mean(network_delays)) if network_delays else 0.,
               "simulated_network_ms_arrived_mean": mean_net_arrivals,
               "simulated_staleness_rounds_mean": mean_stale,
               "local_training_s_sum": sum(compute_times),
               "wall_elapsed_s": time.perf_counter() - t_round,
               "resource_light": tiers.count("light"), "resource_medium": tiers.count("medium"),
               "resource_full": tiers.count("full")}
        rows.append(row)
        # Save EACH round so data survives Colab disconnects.
        write_csv(out / "round_metrics.csv", rows)
        if args.checkpoint_every and rnd % args.checkpoint_every == 0:
            torch.save(global_state, out / f"checkpoint_round_{rnd}.pt")
        print(f"{args.dataset} {method} {ablation} seed={seed} round={rnd}/{args.rounds} "
              f"accuracy={metrics['accuracy']:.4f} uplink={bytes_round:,}B "
              f"vs_dense={row['uplink_reduction_pct_vs_dense']:.1f}% queued={len(pending)}", flush=True)
    torch.save(global_state, out / "final_model.pt")
    return rows[-1]


def parse_args(argv=None):
    parser = argparse.ArgumentParser(description="DW-AFL corrected research runner")
    parser.add_argument("--dataset", choices=["mnist", "cifar10", "synthetic"], default="mnist")
    parser.add_argument("--methods", default="dwafl,fedavg,fedasync,lag,dgc,paflm")
    parser.add_argument("--seeds", default="42")
    parser.add_argument("--clients", type=int, default=10)
    parser.add_argument("--per-round", type=int, default=5)
    parser.add_argument("--rounds", type=int, default=30)
    parser.add_argument("--local-epochs", type=int, default=2)
    parser.add_argument("--batch", type=int, default=32)
    parser.add_argument("--eval-batch", type=int, default=256)
    parser.add_argument("--lr", type=float, default=.001)
    parser.add_argument("--width", type=int, default=16)
    parser.add_argument("--partition", choices=["iid", "dirichlet"], default="dirichlet")
    parser.add_argument("--alpha", type=float, default=.5)
    parser.add_argument("--min-samples", type=int, default=2)
    parser.add_argument("--train-limit", type=int, default=0)
    parser.add_argument("--test-limit", type=int, default=0)
    parser.add_argument("--keep-ratio", type=float, default=.10)
    parser.add_argument("--min-keep-ratio", type=float, default=.03)
    parser.add_argument("--max-keep-ratio", type=float, default=.20)
    parser.add_argument("--ema-decay", type=float, default=.85)
    parser.add_argument("--server-lr", type=float, default=.85)
    parser.add_argument("--staleness-decay", type=float, default=.12)
    parser.add_argument("--max-staleness", type=int, default=2)
    parser.add_argument("--lag-threshold", type=float, default=.05)
    parser.add_argument("--noise", type=float, default=0.)
    parser.add_argument("--clip", type=float, default=10.)
    parser.add_argument("--policy-steps", type=int, default=80)
    parser.add_argument("--warmup-rounds", type=int, default=5)
    parser.add_argument("--bandwidth-min", type=float, default=1.)
    parser.add_argument("--bandwidth-max", type=float, default=10.)
    parser.add_argument("--propagation-ms", type=float, default=10.)
    parser.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    parser.add_argument("--data-dir", default="data")
    parser.add_argument("--output", default="results")
    parser.add_argument("--ablations", action="store_true")
    parser.add_argument("--checkpoint-every", type=int, default=0)
    parser.add_argument("--deterministic", action="store_true")
    args = parser.parse_args(argv)   # FIX: not parse_args(args=[])
    if args.clients < 1 or args.per_round < 1 or args.local_epochs < 1 or args.rounds < 1:
        parser.error("clients, per-round, local-epochs and rounds must all be positive")
    if args.alpha <= 0 or not (0 < args.min_keep_ratio <= args.keep_ratio <= args.max_keep_ratio <= 1):
        parser.error("alpha > 0 and 0 < min_keep <= keep <= max_keep <= 1 are required")
    if args.max_staleness < 0 or not (0 < args.server_lr <= 1):
        parser.error("max-staleness >= 0 and server-lr in (0,1] required")
    methods = [m.strip() for m in args.methods.split(",") if m.strip()]
    if any(m not in METHODS for m in methods):
        parser.error(f"Unknown method in {methods}; valid methods are {METHODS}")
    return args


def main(argv=None):
    args = parse_args(argv)
    seeds = [int(k.strip()) for k in args.seeds.split(",")]
    methods = [k.strip() for k in args.methods.split(",") if k.strip()]
    summaries = []
    for seed in seeds:
        for method in methods:
            summaries.append(run_experiment(args, method, seed, "full"))
            write_csv(Path(args.output) / f"summary_{args.dataset}.csv", summaries)
            if method == "dwafl" and args.ablations:
                for variant in ABLATIONS:
                    if variant == "no_noise" and args.noise == 0:
                        print("Skipping no_noise ablation: noise is zero in the full configuration.")
                        continue
                    summaries.append(run_experiment(args, method, seed, variant))
                    write_csv(Path(args.output) / f"summary_{args.dataset}.csv", summaries)
    print("\nExperiment complete; see", Path(args.output).resolve())
    return summaries

fast_args = ['--dataset', 'cifar10', '--methods', 'dwafl', '--clients', '4', '--per-round', '2', '--rounds', '8', '--local-epochs', '1', '--batch', '64', '--eval-batch', '256', '--lr', '0.001', '--width', '12', '--partition', 'dirichlet', '--alpha', '0.5', '--train-limit', '1600', '--test-limit', '400', '--policy-steps', '20', '--warmup-rounds', '2', '--max-staleness', '1', '--output', 'cifar10_fast_results']
results = main(argv=fast_args)
print("\nFinal accuracy:", round(100 * results[-1]["accuracy"], 2), "%")
print("Last-round uplink bytes:", results[-1]["uplink_bytes"])
print("Saved at: cifar10_fast_results/cifar10/dwafl/seed_42/full/round_metrics.csv")


cifar10 dwafl full seed=42 round=1/8 accuracy=0.0925 uplink=26,024B vs_dense=80.0% queued=1
cifar10 dwafl full seed=42 round=2/8 accuracy=0.0925 uplink=25,864B vs_dense=80.1% queued=1
cifar10 dwafl full seed=42 round=3/8 accuracy=0.0925 uplink=25,536B vs_dense=80.4% queued=1
cifar10 dwafl full seed=42 round=4/8 accuracy=0.0925 uplink=25,376B vs_dense=80.5% queued=0
cifar10 dwafl full seed=42 round=5/8 accuracy=0.0925 uplink=26,064B vs_dense=79.9% queued=2
cifar10 dwafl full seed=42 round=6/8 accuracy=0.0925 uplink=26,064B vs_dense=79.9% queued=1
cifar10 dwafl full seed=42 round=7/8 accuracy=0.0925 uplink=25,824B vs_dense=80.1% queued=1
cifar10 dwafl full seed=42 round=8/8 accuracy=0.0925 uplink=25,336B vs_dense=80.5% queued=1

Experiment complete; see /content/cifar10_fast_results

Final accuracy: 9.25 %
Last-round uplink bytes: 25336
Saved at: cifar10_fast_results/cifar10/dwafl/seed_42/full/round_metrics.csv
